# SHAP Analysis Notebook

This notebook corresponds to `Untitled.ipynb` Cell 10.

It computes SHAP values for sampled label=1 and label=0 sequences and saves the pkl files used by later SHAP plotting/statistics. Plotting is handled separately.

## Step 0. Project root

In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "lamar_m6a").exists():
    raise RuntimeError("Cannot find project root containing lamar_m6a/. Run this notebook inside the project directory.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/picb/rnasys2/lidaoqian/LAMAR-M6A')

## Step 1. Import analysis function

In [3]:
from lamar_m6a.interpretability.run_shap_analysis import run_shap_analysis
from argparse import Namespace

/opt/anaconda3/envs/torch201/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Step 2. Configure inputs and outputs

The input CSV should contain `seq` and `label`. SHAP can be slow; start with a small `MAX_SEQUENCES` for testing.

In [ ]:
MODE = "full"
TEST_CELL = "cell5"      # 被分析的测试集来自哪个 cell
MODEL_CELL = "cell5"      # 使用哪个 fine-tuned 模型
LENGTH = 801
MAX_SEQUENCES = 500

INPUT_CSV = PROJECT_ROOT / "outputs" / "predictions" / "single_csv" / MODE / f"model_{MODEL_CELL}_target_{TEST_CELL}_{LENGTH}_test_pred.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "checkpoints" / MODE / f"{MODEL_CELL}-{LENGTH}-bs16-acc4"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interpretability" / "shap" / MODE / f"{TEST_CELL}_{LENGTH}"
LABEL1_OUTPUT = OUTPUT_DIR / f"label1_test_{MAX_SEQUENCES}.pkl"
LABEL0_OUTPUT = OUTPUT_DIR / f"label0_test_{MAX_SEQUENCES}.pkl"

TOKENIZER_PATH = PROJECT_ROOT / "tokenizer" / "single_nucleotide"
MODEL_CONFIG = PROJECT_ROOT / "config" / "config_150M.json"
MODEL_CLASS = "scripts.sequence_classification_patch:EsmForSequenceClassification"

INPUT_CSV, LABEL1_OUTPUT, LABEL0_OUTPUT

## Step 3. Check paths

In [ ]:
for path in [INPUT_CSV, CHECKPOINT_PATH, TOKENIZER_PATH, MODEL_CONFIG]:
    print(path, path.exists())

## Step 4. Run SHAP analysis

In [ ]:
args = Namespace(
    input_csv=INPUT_CSV,
    checkpoint_path=CHECKPOINT_PATH,
    label1_output=LABEL1_OUTPUT,
    label0_output=LABEL0_OUTPUT,
    max_sequences=MAX_SEQUENCES,
    length=LENGTH,
    model_max_length=LENGTH + 2,
    class_index=1,
    nsamples=512,
    batch_size=1,
    background_size=1,
    seed=42,
    tokenizer_path=TOKENIZER_PATH,
    model_config=MODEL_CONFIG,
    model_class=MODEL_CLASS,
    padding_side="left",
    use_fast_tokenizer=False,
    num_labels=2,
    token_dropout=False,
    positional_embedding_type="rotary",
    hidden_size=768,
    intermediate_size=3072,
    num_attention_heads=12,
    num_hidden_layers=12,
    head_type="Linear",
    kernel_sizes=None,
    ocs=None,
    strict_load=True,
    device=None,
)

run_shap_analysis(args)

## Step 5. Output check

In [ ]:
import pickle

for path in [LABEL1_OUTPUT, LABEL0_OUTPUT]:
    print(path, path.exists())
    if path.exists():
        with open(path, "rb") as f:
            values = pickle.load(f)
        print("records:", len(values))
        if values:
            print("first shape:", getattr(values[0], "shape", None))